# Clase 9 — Hashing de Contraseñas, KDFs y Gestión de Secretos

> **Curso:** Criptografía Aplicada  \n> **Objetivo general:** Diseñar almacenamiento de credenciales robusto frente a ataques offline y derivar claves de forma segura para cifrado, autenticación y gestión de secretos.

---

## Tabla de contenidos

1. [Introducción y objetivos](#1)
2. [Modelo de amenaza para credenciales](#2)
3. [Ataques: diccionario, fuerza bruta y rainbow tables](#3)
4. [Por qué no usar hash rápido para contraseñas](#4)
5. [Sales, pepper y parámetros de costo](#5)
6. [PBKDF2 en profundidad](#6)
7. [scrypt en profundidad](#7)
8. [bcrypt en profundidad](#8)
9. [Argon2 en profundidad](#9)
10. [Comparativa práctica de KDF/password hashing](#10)
11. [Verificación y migración de hashes heredados](#11)
12. [HKDF para derivación de claves](#12)
13. [Gestión de secretos en sistemas reales](#13)
14. [Checklist de hardening y monitoreo](#14)
15. [Temas adicionales pertinentes](#15)
16. [Laboratorio integrado](#16)
17. [Ejercicios propuestos](#17)
18. [Referencias](#18)


<a id='1'></a>
## 1) Introducción y objetivos

En seguridad aplicada, **proteger contraseñas** y **derivar claves** son problemas distintos pero relacionados:

- **Password hashing**: almacenar verificadores resistentes a cracking offline.
- **KDFs criptográficas**: derivar claves de alta calidad para cifrado/MAC/AEAD.

En esta clase aprenderás a:

1. Modelar amenazas realistas ante fuga de base de datos.
2. Implementar y comparar **PBKDF2, scrypt, bcrypt y Argon2**.
3. Entender el rol de **salt**, **pepper**, memoria y paralelismo.
4. Aplicar **HKDF** para expandir secretos de entrada en múltiples claves con propósito.
5. Diseñar flujos de **gestión de secretos** en aplicaciones modernas.


<a id='2'></a>
## 2) Modelo de amenaza para credenciales

Supongamos que un atacante obtiene la tabla de usuarios:

- `username`
- `password_hash`
- `salt`
- `parámetros del algoritmo`

En escenarios realistas el atacante puede:

- Ejecutar cracking offline sin límite de intentos.
- Usar diccionarios gigantes y reglas de mutación.
- Paralelizar en CPU/GPU/ASIC.

### Propiedad clave que buscamos

Cada verificación de contraseña debe ser **costosa** en tiempo y/o memoria para elevar el costo por intento del atacante.


## Importaciones globales

Usaremos principalmente librerías estándar de Python para que el notebook sea ejecutable en la mayoría de entornos.
Se agregarán secciones opcionales para `bcrypt` y `argon2-cffi` si están disponibles.


In [ ]:
import os
import time
import hmac
import json
import base64
import secrets
import hashlib
import statistics
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any

print("Entorno listo ✓")


<a id='3'></a>
## 3) Ataques: diccionario, fuerza bruta y rainbow tables

### 3.1 Ataque de diccionario

Prueba contraseñas probables (`123456`, `qwerty`, `password`, etc.) contra cada hash filtrado.

### 3.2 Fuerza bruta

Explora todo el espacio de búsqueda según alfabeto y longitud.

### 3.3 Rainbow tables

Son tablas precomputadas de hash(sin sal) -> contraseña.
Su ventaja colapsa cuando cada contraseña usa una **salt aleatoria** y única.


In [ ]:
# Demo educativa: ataque de diccionario sobre SHA-256 sin sal (mala práctica)
users_plain = {
    "ana": "password123",
    "bruno": "letmein",
    "carla": "S3gura?2026"
}

leaked_fast_hash_db = {u: hashlib.sha256(p.encode()).hexdigest() for u, p in users_plain.items()}
print("Base filtrada (SHA-256 sin sal):")
for u, h in leaked_fast_hash_db.items():
    print(f"  {u}: {h[:20]}...")

common_passwords = [
    "123456", "password", "password123", "qwerty", "letmein",
    "admin", "welcome", "iloveyou", "123456789", "dragon"
]

recovered = {}
for user, digest in leaked_fast_hash_db.items():
    for guess in common_passwords:
        if hashlib.sha256(guess.encode()).hexdigest() == digest:
            recovered[user] = guess
            break

print("\nCredenciales recuperadas por diccionario:")
print(recovered)


In [ ]:
# Demo educativa: mini-rainbow table para hashes sin sal
mini_space = ["1234", "admin", "letmein", "password123", "abc123"]
rainbow = {hashlib.sha256(p.encode()).hexdigest(): p for p in mini_space}

victim_hash = leaked_fast_hash_db["ana"]
print("Hash víctima:", victim_hash)
print("Lookup directo en rainbow table:", rainbow.get(victim_hash, "no encontrado"))

print("\nConclusión: sin salt, un precomputo sirve para muchas víctimas.")


<a id='4'></a>
## 4) Por qué no usar hash rápido para contraseñas

Algoritmos como SHA-256/SHA-512/BLAKE2 son excelentes para integridad, pero para contraseñas son **demasiado rápidos**.

Un atacante con hardware especializado puede probar enormes volúmenes por segundo.
Necesitamos funciones con costo ajustable (iteraciones, memoria, paralelismo).


In [ ]:
# Benchmark rápido: SHA-256 vs PBKDF2 (didáctico)
password = b"S3gura?2026"
salt = secrets.token_bytes(16)

N = 120_000

t0 = time.perf_counter()
for _ in range(N):
    hashlib.sha256(password).digest()
sha_time = time.perf_counter() - t0

t1 = time.perf_counter()
for _ in range(500):
    hashlib.pbkdf2_hmac("sha256", password, salt, 200_000, dklen=32)
pbkdf2_time = time.perf_counter() - t1

print(f"SHA-256: {N} operaciones en {sha_time:.4f}s")
print(f"PBKDF2: 500 derivaciones en {pbkdf2_time:.4f}s")
print("(comparación orientativa: no extrapolar linealmente entre máquinas)")


<a id='5'></a>
## 5) Sales, pepper y parámetros de costo

### 5.1 Salt

- Aleatoria por usuario (mínimo 16 bytes).
- Se almacena junto al hash.
- Evita colisiones de hash entre usuarios con misma contraseña.
- Inutiliza rainbow tables globales.

### 5.2 Pepper

- Secreto adicional global o por dominio, guardado fuera de la base (HSM/Vault/env seguro).
- Si roban DB pero no pepper, el cracking se vuelve más difícil.

### 5.3 Costo

- Debe calibrarse para que login sea aceptable para usuario legítimo.
- A la vez, suficientemente caro para atacantes offline.
- Se recomienda revisar parámetros periódicamente.


In [ ]:
# Ejemplo de salt + pepper + PBKDF2
APP_PEPPER = os.environ.get("APP_PEPPER", "DEMO_PEPPER_NO_PROD").encode()

def pbkdf2_hash_password(password: str, iterations: int = 300_000) -> Dict[str, Any]:
    salt = secrets.token_bytes(16)
    pwd_bytes = password.encode("utf-8") + APP_PEPPER
    dk = hashlib.pbkdf2_hmac("sha256", pwd_bytes, salt, iterations, dklen=32)
    return {
        "algorithm": "pbkdf2-sha256",
        "iterations": iterations,
        "salt_b64": base64.b64encode(salt).decode(),
        "hash_b64": base64.b64encode(dk).decode(),
    }

def pbkdf2_verify_password(password: str, record: Dict[str, Any]) -> bool:
    salt = base64.b64decode(record["salt_b64"])
    expected = base64.b64decode(record["hash_b64"])
    pwd_bytes = password.encode("utf-8") + APP_PEPPER
    candidate = hashlib.pbkdf2_hmac("sha256", pwd_bytes, salt, int(record["iterations"]), dklen=len(expected))
    return hmac.compare_digest(candidate, expected)

record = pbkdf2_hash_password("S3gura?2026")
print(json.dumps(record, indent=2))
print("verify correcta:", pbkdf2_verify_password("S3gura?2026", record))
print("verify incorrecta:", pbkdf2_verify_password("badpass", record))


<a id='6'></a>
## 6) PBKDF2 en profundidad

**PBKDF2** (RFC 8018) usa HMAC + iteraciones para ralentizar ataques.

Ventajas:
- Amplia disponibilidad y estandarización.
- Fácil interoperabilidad.

Limitación:
- Es principalmente CPU-hard, no memory-hard.

Uso recomendado:
- Sistemas legacy o entornos donde Argon2/scrypt no estén disponibles.


In [ ]:
# Calibración aproximada de iteraciones PBKDF2 para un presupuesto de tiempo objetivo
def calibrate_pbkdf2(target_ms: float = 120.0, max_iter: int = 2_000_000) -> int:
    password = b"calibration-password"
    salt = secrets.token_bytes(16)
    it = 50_000
    while it <= max_iter:
        t0 = time.perf_counter()
        hashlib.pbkdf2_hmac("sha256", password, salt, it, dklen=32)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        if elapsed_ms >= target_ms:
            return it
        it = int(it * 1.35)
    return max_iter

recommended_iter = calibrate_pbkdf2(target_ms=120)
print("Iteraciones PBKDF2 recomendadas para ~120ms en esta máquina:", recommended_iter)


<a id='7'></a>
## 7) scrypt en profundidad

**scrypt** está diseñado para ser **memory-hard**: además de CPU, consume memoria significativa.

Parámetros típicos:
- `n` (coste CPU/memoria, potencia de 2),
- `r` (block size),
- `p` (paralelismo).

Mayor memoria requerida => menor ventaja de GPU/ASIC para cracking masivo.


In [ ]:
def scrypt_hash_password(password: str, n: int = 2**14, r: int = 8, p: int = 1) -> Dict[str, Any]:
    salt = secrets.token_bytes(16)
    dk = hashlib.scrypt(password.encode(), salt=salt, n=n, r=r, p=p, dklen=32)
    return {
        "algorithm": "scrypt",
        "n": n,
        "r": r,
        "p": p,
        "salt_b64": base64.b64encode(salt).decode(),
        "hash_b64": base64.b64encode(dk).decode(),
    }

def scrypt_verify_password(password: str, record: Dict[str, Any]) -> bool:
    salt = base64.b64decode(record["salt_b64"])
    expected = base64.b64decode(record["hash_b64"])
    candidate = hashlib.scrypt(
        password.encode(),
        salt=salt,
        n=int(record["n"]),
        r=int(record["r"]),
        p=int(record["p"]),
        dklen=len(expected),
    )
    return hmac.compare_digest(candidate, expected)

srec = scrypt_hash_password("S3gura?2026")
print(json.dumps(srec, indent=2))
print("verify correcta:", scrypt_verify_password("S3gura?2026", srec))
print("verify incorrecta:", scrypt_verify_password("badpass", srec))


In [ ]:
# Benchmark comparativo simple: PBKDF2 vs scrypt
def bench(func, runs=5):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        func()
        times.append((time.perf_counter() - t0) * 1000)
    return {
        "min_ms": min(times),
        "max_ms": max(times),
        "mean_ms": statistics.mean(times),
        "runs": runs
    }

pwd = "benchmark-pass-2026"

pbkdf2_stats = bench(lambda: hashlib.pbkdf2_hmac("sha256", pwd.encode(), b"0"*16, 300_000, dklen=32), runs=6)
scrypt_stats = bench(lambda: hashlib.scrypt(pwd.encode(), salt=b"1"*16, n=2**14, r=8, p=1, dklen=32), runs=6)

print("PBKDF2 stats:", pbkdf2_stats)
print("scrypt stats:", scrypt_stats)


<a id='8'></a>
## 8) bcrypt en profundidad

**bcrypt** incorpora salt y costo (`2^cost`) de forma nativa, y fue ampliamente adoptado durante años.

Fortalezas:
- Madurez y ecosistema.
- Formato de hash estándar con parámetros embebidos.

Precauciones:
- Límite práctico de longitud de entrada en algunas implementaciones.
- Hoy suele preferirse Argon2id para sistemas nuevos, cuando está disponible.


In [ ]:
# Sección opcional bcrypt (si la dependencia está instalada)
try:
    import bcrypt
    BCRYPT_OK = True
except Exception as e:
    BCRYPT_OK = False
    print("bcrypt no disponible:", e)

if BCRYPT_OK:
    password = b"S3gura?2026"
    salt = bcrypt.gensalt(rounds=12)
    h = bcrypt.hashpw(password, salt)
    print("Hash bcrypt:", h.decode())
    print("verify correcta:", bcrypt.checkpw(password, h))
    print("verify incorrecta:", bcrypt.checkpw(b"badpass", h))
else:
    print("Instalación sugerida: pip install bcrypt")


<a id='9'></a>
## 9) Argon2 en profundidad

**Argon2** (PHC winner) es el estándar moderno recomendado para password hashing.

Versiones:
- Argon2d (resistencia GPU, menos resistente side-channel),
- Argon2i (más resistente side-channel),
- **Argon2id** (híbrida recomendada para uso general).

Parámetros clave:
- `time_cost` (iteraciones),
- `memory_cost` (KiB),
- `parallelism` (hilos).


In [ ]:
# Sección opcional Argon2 (si la dependencia está instalada)
try:
    from argon2 import PasswordHasher
    from argon2.exceptions import VerifyMismatchError
    ARGON2_OK = True
except Exception as e:
    ARGON2_OK = False
    print("argon2-cffi no disponible:", e)

if ARGON2_OK:
    ph = PasswordHasher(
        time_cost=3,
        memory_cost=64 * 1024,  # 64 MiB
        parallelism=2,
        hash_len=32,
        salt_len=16,
    )
    h = ph.hash("S3gura?2026")
    print("Hash Argon2:", h)

    try:
        print("verify correcta:", ph.verify(h, "S3gura?2026"))
    except VerifyMismatchError:
        print("verify correcta: False")

    try:
        ph.verify(h, "badpass")
        print("verify incorrecta: True (inesperado)")
    except VerifyMismatchError:
        print("verify incorrecta: False")

    print("¿Rehash recomendado?:", ph.check_needs_rehash(h))
else:
    print("Instalación sugerida: pip install argon2-cffi")


<a id='10'></a>
## 10) Comparativa práctica de KDF/password hashing

No existe parámetro universal. Se debe calibrar por entorno (hardware, SLA, UX, carga concurrente).

Objetivo común en autenticación web:
- Verificación en orden de decenas a pocos cientos de milisegundos por intento legítimo.


In [ ]:
comparison = {
    "PBKDF2": {
        "tipo": "CPU-hard",
        "estandar": "RFC 8018",
        "fortaleza": "Compatibilidad amplia",
        "debilidad": "Menor resistencia GPU que memory-hard"
    },
    "scrypt": {
        "tipo": "Memory-hard",
        "estandar": "RFC 7914",
        "fortaleza": "Dificulta cracking masivo",
        "debilidad": "Más sensible a tuning de memoria"
    },
    "bcrypt": {
        "tipo": "CPU-hard (Blowfish-based)",
        "estandar": "De facto",
        "fortaleza": "Muy adoptado históricamente",
        "debilidad": "Menos flexible que Argon2"
    },
    "Argon2id": {
        "tipo": "Memory-hard moderno",
        "estandar": "RFC 9106",
        "fortaleza": "Recomendado para nuevos despliegues",
        "debilidad": "Puede requerir dependencia adicional"
    }
}

print(json.dumps(comparison, indent=2, ensure_ascii=False))


<a id='11'></a>
## 11) Verificación y migración de hashes heredados

Escenario común: usuarios antiguos con SHA-1/SHA-256 sin costo.

Estrategia de migración segura:
1. Soportar temporalmente validación legacy.
2. Si login legacy es correcto, **rehash inmediato** con algoritmo moderno.
3. Guardar nuevo hash y eliminar formato antiguo.
4. Forzar reset para cuentas inactivas de alto riesgo.


In [ ]:
# Demo de migración on-login: legacy SHA-256 -> PBKDF2
legacy_db = {
    "diego": {"scheme": "sha256", "hash": hashlib.sha256(b"diego123").hexdigest()}
}

new_db = {}

def login_and_migrate(username: str, password: str) -> bool:
    if username in new_db:
        return pbkdf2_verify_password(password, new_db[username])

    rec = legacy_db.get(username)
    if not rec:
        return False

    if rec["scheme"] == "sha256":
        if hashlib.sha256(password.encode()).hexdigest() == rec["hash"]:
            new_db[username] = pbkdf2_hash_password(password, iterations=300_000)
            del legacy_db[username]
            return True
    return False

print("Antes:", {"legacy": list(legacy_db.keys()), "new": list(new_db.keys())})
print("Login correcto:", login_and_migrate("diego", "diego123"))
print("Después:", {"legacy": list(legacy_db.keys()), "new": list(new_db.keys())})


<a id='12'></a>
## 12) HKDF para derivación de claves

Cuando ya tenemos un secreto de entrada (por ejemplo, ECDH compartido o key material maestro), usamos **HKDF** (RFC 5869):

- `Extract`: normaliza entropía en una PRK.
- `Expand`: deriva subclaves separadas por contexto (`info`).

Esto permite *key separation* (claves distintas para cifrado, MAC, nonce derivation, etc.).


In [ ]:
def hkdf_extract(salt: bytes, ikm: bytes, hashmod=hashlib.sha256) -> bytes:
    return hmac.new(salt, ikm, hashmod).digest()

def hkdf_expand(prk: bytes, info: bytes, length: int, hashmod=hashlib.sha256) -> bytes:
    out = b""
    t = b""
    counter = 1
    while len(out) < length:
        t = hmac.new(prk, t + info + bytes([counter]), hashmod).digest()
        out += t
        counter += 1
    return out[:length]

master_input = secrets.token_bytes(32)
salt = secrets.token_bytes(16)
prk = hkdf_extract(salt, master_input)

k_enc = hkdf_expand(prk, b"app:v1:encryption", 32)
k_mac = hkdf_expand(prk, b"app:v1:authentication", 32)
k_wrap = hkdf_expand(prk, b"app:v1:key-wrap", 32)

print("PRK:", prk.hex())
print("k_enc:", k_enc.hex())
print("k_mac:", k_mac.hex())
print("k_wrap:", k_wrap.hex())
print("¿Claves distintas?:", len({k_enc, k_mac, k_wrap}) == 3)


## 12.1 Error común: reutilizar una sola clave para múltiples propósitos

Buenas prácticas:
- Usar HKDF con `info` específico de contexto.
- Versionar etiquetas (`app:v1:...`) para rotaciones controladas.
- Mantener separación de dominios criptográficos.


<a id='13'></a>
## 13) Gestión de secretos en sistemas reales

La seguridad no termina en el algoritmo. También importa dónde viven los secretos:

- Variables de entorno (mínimo viable, riesgo por exposición de proceso).
- Secret managers (Vault, AWS Secrets Manager, GCP Secret Manager, Azure Key Vault).
- HSM/KMS para operaciones con claves maestras.

### Principios

1. **Least privilege**: cada servicio accede sólo a lo que necesita.
2. **Rotación** periódica y por incidente.
3. **Auditoría** de acceso a secretos.
4. **Separación** entre datos y material secreto (pepper, KEKs).
5. **No hardcodear** secretos en repositorio ni imágenes.


In [ ]:
# Ejemplo simple de "secret bootstrap" desde entorno
# (En producción: preferir gestor de secretos con identidad de workload.)

required_vars = ["APP_PEPPER", "JWT_SIGNING_KEY"]
missing = [v for v in required_vars if not os.environ.get(v)]

if missing:
    print("Faltan secretos de entorno:", missing)
    print("Demo: continuar en modo educativo con valores no seguros.")
else:
    print("Todos los secretos requeridos presentes.")


<a id='14'></a>
## 14) Checklist de hardening y monitoreo

- [ ] Algoritmo moderno (Argon2id o scrypt; PBKDF2 si compatibilidad lo exige).
- [ ] Salt aleatoria por credencial.
- [ ] Pepper externa a la DB.
- [ ] Comparación constante (`hmac.compare_digest`).
- [ ] Rate limiting + lockout progresivo + MFA.
- [ ] Telemetría de intentos fallidos y alertas de anomalías.
- [ ] Rotación de secretos y playbooks de respuesta a incidentes.
- [ ] Política de contraseñas + bloqueo de credenciales comprometidas.


<a id='15'></a>
## 15) Temas adicionales pertinentes

1. **PAKE (SRP, OPAQUE)**: autenticación resistente a phishing y fuga de verificadores.
2. **WebAuthn/FIDO2**: reducción drástica de dependencia de contraseñas.
3. **Credential stuffing defense**: reputación de IP, device fingerprint, bot detection.
4. **Secret zero problem**: cómo inicia confianza un workload al arrancar.
5. **Post-quantum considerations**: impacto limitado en password hashing vs impacto alto en PKI.


<a id='16'></a>
## 16) Laboratorio integrado

Construiremos un mini "servicio de autenticación" en memoria que:
- registra usuarios con PBKDF2 o scrypt,
- verifica credenciales,
- permite rehash cuando cambian parámetros objetivo.


In [ ]:
@dataclass
class UserRecord:
    username: str
    scheme: str
    payload: Dict[str, Any]

class AuthStore:
    def __init__(self):
        self.users: Dict[str, UserRecord] = {}

    def register_pbkdf2(self, username: str, password: str, iterations: int = 300_000):
        self.users[username] = UserRecord(username, "pbkdf2", pbkdf2_hash_password(password, iterations))

    def register_scrypt(self, username: str, password: str, n: int = 2**14, r: int = 8, p: int = 1):
        self.users[username] = UserRecord(username, "scrypt", scrypt_hash_password(password, n=n, r=r, p=p))

    def verify(self, username: str, password: str) -> bool:
        rec = self.users.get(username)
        if not rec:
            return False
        if rec.scheme == "pbkdf2":
            return pbkdf2_verify_password(password, rec.payload)
        if rec.scheme == "scrypt":
            return scrypt_verify_password(password, rec.payload)
        return False

    def needs_rehash_pbkdf2(self, username: str, target_iterations: int) -> bool:
        rec = self.users.get(username)
        if not rec or rec.scheme != "pbkdf2":
            return False
        return int(rec.payload.get("iterations", 0)) < target_iterations

    def rehash_pbkdf2(self, username: str, password: str, target_iterations: int):
        if self.verify(username, password):
            self.users[username] = UserRecord(username, "pbkdf2", pbkdf2_hash_password(password, target_iterations))
            return True
        return False

store = AuthStore()
store.register_pbkdf2("alice", "Alice#2026", iterations=180_000)
store.register_scrypt("bob", "Bob#2026")

print("alice login ok:", store.verify("alice", "Alice#2026"))
print("alice login fail:", store.verify("alice", "bad"))
print("bob login ok:", store.verify("bob", "Bob#2026"))

print("alice necesita rehash a 300k?:", store.needs_rehash_pbkdf2("alice", 300_000))
print("rehash ejecutado:", store.rehash_pbkdf2("alice", "Alice#2026", 300_000))
print("alice necesita rehash ahora?:", store.needs_rehash_pbkdf2("alice", 300_000))


<a id='17'></a>
## 17) Ejercicios propuestos

1. Ajusta PBKDF2 para objetivo de 200 ms y compara UX entre desktop y servidor cloud.
2. Repite benchmark de scrypt variando `n`, `r`, `p`; grafica tiempo y consumo de memoria.
3. Implementa política de bloqueo progresivo tras fallos y evalúa riesgo de DoS.
4. Añade un flujo de "password reset" con token firmado y expiración.
5. Implementa versionado de esquema de hash y migración automática multi-etapa.
6. Extiende HKDF para derivar claves por tenant (`tenant_id` en `info`).
7. Diseña un playbook de respuesta ante fuga de base de credenciales.


<a id='18'></a>
## 18) Referencias

### Estándares y RFCs
- RFC 8018 — PKCS #5: Password-Based Cryptography Specification (PBKDF2).
- RFC 7914 — The scrypt Password-Based Key Derivation Function.
- RFC 9106 — Argon2 Memory-Hard Function for Password Hashing and Proof-of-Work Applications.
- RFC 5869 — HMAC-based Extract-and-Expand Key Derivation Function (HKDF).

### Guías de seguridad
- OWASP Password Storage Cheat Sheet.
- OWASP Secrets Management Cheat Sheet.
- NIST SP 800-63B — Digital Identity Guidelines (Authentication and Lifecycle Management).

### Lecturas recomendadas
- Password Hashing Competition (PHC) design paper and Argon2 docs.
- Academic literature sobre cracking GPU y economía de ataques offline.
- Documentación oficial de bibliotecas `argon2-cffi` y `bcrypt`.

---

## Cierre

La defensa efectiva de credenciales depende de combinar:

- algoritmo adecuado,
- parámetros correctos para tu hardware,
- operación segura de secretos,
- monitoreo y respuesta continua.

No existe una configuración "eterna": la seguridad aquí es un proceso de ajuste permanente.
